# OOF 3-Fold Training — Qwen3.6-35B-A3B

3개 fold 순차 학습 → lora_model_fold_0, fold_1, fold_2 생성

학습 후 **colab_inference_ensemble.ipynb** 에서 앙상블 추론

## 1. Install Dependencies
**실행 후 Runtime > Restart Session 필수**

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --upgrade "transformers==5.5.0" "trl==0.24.0" "datasets<4.4.0"
!pip install cut_cross_entropy hf_transfer msgspec tyro peft accelerate bitsandbytes xformers
!pip install flash-attn --no-build-isolation
!pip install pandas tqdm scikit-learn sentence-transformers

## 2. 파일 업로드 확인
왼쪽 파일 패널에서 `my_code_0514from0508/` 폴더와 `data/` 폴더 업로드 후 실행

In [ ]:
import os
os.chdir('/content')

SRC_DIR = '/content/my_code_0514from0508'  # 업로드한 폴더명

print('src files:', os.listdir(SRC_DIR))
print('data files:', os.listdir('/content/data'))

## 3. GPU 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 4. OOF 3-Fold 학습 실행
약 3~4시간 소요 (A100 기준 fold당 1~1.5시간)

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, f'{SRC_DIR}/train.py', '--oof'],
    cwd='/content',
)
print('Return code:', result.returncode)
# 완료 후 확인
!ls /content/lora_model_fold_*

## 5. Fold 모델 압축 다운로드

In [ ]:
from google.colab import files
!zip -r lora_oof_folds.zip lora_model_fold_0/ lora_model_fold_1/ lora_model_fold_2/
files.download('lora_oof_folds.zip')